# 02 · Final koşular (5 seed) + verimlilik ölçümleri
Seçilen hiperparametrelerle her model × `FINAL_SEEDS` (101, 202, 303, 404, 505). Seçim ve erken durma NN val ile; test her `TEST_EVERY` epoch'ta
**yalnız tanı amaçlı** öğrenme eğrisine yazılır. Önerilen model için test anı ablasyonları (η = 0, ohmik kol, tek RC kolu, R(T) sabit) da hesaplanır.
Kayıt: ağırlıklar, JSON (hiperparametre + öğrenme eğrisi + süreler + metrikler), zaman serileri (parquet) ve en iyi seed'in serileri (Excel).

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)  False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
sel = exp.load_selected()
MODELS = [m for m in BASE_SPECS if m in sel]
for name in MODELS:
    for seed in exp.cfg['FINAL_SEEDS']:
        exp.run(name, sel[name], seed, 'final', series=True, ablations=RC_ABLATIONS_TEST if sel[name]['family'] == 'rc' else ())
final = exp.collect('final')
t = (final.assign(k=np.where(final.temp == '-20degC', '−20°C', 'eğitim sıc.')).groupby(['model', 'k', 'seed']).rmse.mean() * 100).groupby(['model', 'k']).agg(['mean', 'std']).unstack('k')
print(t.round(3).to_string())

In [ ]:
# Verimlilik: parametre, boyut, MAC, GPU/CPU tek pencere gecikmesi, bellek, eğitim süresi
eff = []
for name in MODELS:
    m, rec = exp.load_model('final', name, exp.cfg['FINAL_SEEDS'][0])
    X = next(iter(exp.datasets(rec['spec']['window'])['test'].values())).X[:rec['spec']['window']].unsqueeze(0)
    e = engine.efficiency(m, X, n_rep=50 if exp.smoke else 200)
    recs = [r for r in exp.records('final') if r['name'] == name]
    e.update({'model': name, 'W (s)': rec['spec']['window'], 'eğitim süresi ort (dk)': np.mean([r['train']['train_time_s'] for r in recs]) / 60,
              'epoch süresi ort (s)': np.mean([r['train']['epoch_time_mean_s'] for r in recs]),
              'eğitim GPU bellek tepe (GB)': np.mean([r['train']['train_peak_mem_gb'] for r in recs]),
              'ön işleme (ms / örnek)': exp.pre_ms})
    eff.append(e); print(name, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in e.items() if k != 'model'})
eff = pd.DataFrame(eff); eff.to_json(exp.p['configs'] / 'verimlilik.json', orient='records', force_ascii=False, indent=1)

In [ ]:
# En iyi seed'in (en düşük NN val) tüm zaman serileri → Excel (her test dosyası ayrı sayfa)
for name in MODELS:
    recs = [r for r in exp.records('final') if r['name'] == name]
    best = min(recs, key=lambda r: r['val_rmse_final']); _, _, pq = exp.paths('final', name, best['seed'])
    ser = load_table(pq)
    with pd.ExcelWriter(exp.p['excel'] / f"seriler_{SHORT[name]}_s{best['seed']}.xlsx") as xw:
        for (t, c), g in ser.groupby(['temp', 'cycle'], sort=False):
            g.to_excel(xw, sheet_name=f'{t} {c}'[:31], index=False)
    print(f"✅ {name}: seed {best['seed']} serileri yazıldı")